# 13 RAGAS Evaluation

Το notebook εκτελεί την αξιολόγηση RAGAS για τα διαθέσιμα QA outputs και συγκεντρώνει τις μετρικές context precision, context recall, faithfulness και answer relevancy.


In [ ]:

print("RAGAS dependencies are expected from requirements.txt; no inline pip install is run.")


In [ ]:

print("Embedding dependencies are expected from requirements.txt; no inline pip install is run.")


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd


In [ ]:
# ============================================================
# Ρυθμίσεις εκτέλεσης
# ============================================================
RUN_NAMES = ["dense", "hybrid", "hybrid_reranked"]

# None: πλήρης εκτέλεση (150 ερωτήσεις x 3 runs).
# Για δοκιμαστική εκτέλεση μπορεί να οριστεί μικρότερο δείγμα.
RAGAS_SAMPLE_LIMIT = None

SAVE_OUTPUTS = True

print(f"Εκτελέσεις: {RUN_NAMES}")
print(f"Δείγμα: {RAGAS_SAMPLE_LIMIT or 'πλήρες δείγμα (150 ανά run)'}")
print(f"Αποθήκευση: {SAVE_OUTPUTS}")


In [ ]:

# ============================================================
# Διαδρομές — local-first, με δυναμικό Kaggle fallback
# ============================================================
CURRENT_DIR = Path.cwd()
BASE_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

LOCAL_QA_DIR = BASE_DIR / "data" / "processed" / "qa_results"
LOCAL_EVAL_DIR = BASE_DIR / "data" / "processed" / "evaluation"

QA_DIR = LOCAL_QA_DIR
EVAL_DIR = LOCAL_EVAL_DIR

local_dense_path = LOCAL_QA_DIR / "rag_qa_results_dense.csv"
if not local_dense_path.exists() and Path("/kaggle/input").exists():
    attached_dense_paths = sorted(
        Path("/kaggle/input").glob("**/rag_qa_results_dense.csv")
    )
    if attached_dense_paths:
        QA_DIR = attached_dense_paths[0].parent

DENSE_QA_PATH  = QA_DIR / "rag_qa_results_dense.csv"
HYBRID_QA_PATH = QA_DIR / "rag_qa_results_hybrid.csv"
RERANK_QA_PATH = QA_DIR / "rag_qa_results_hybrid_reranked.csv"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
RAGAS_RESULTS_PATH = EVAL_DIR / "ragas_results.csv"
RAGAS_DETAILS_PATH = EVAL_DIR / "ragas_detailed_results.csv"

print("QA_DIR:", QA_DIR)
print("EVAL_DIR:", EVAL_DIR)
print("Έλεγχος διαδρομών εισόδου...")
for name, p in [
    ("dense QA",  DENSE_QA_PATH),
    ("hybrid QA", HYBRID_QA_PATH),
    ("rerank QA", RERANK_QA_PATH),
]:
    status = "OK" if p.exists() else "δεν βρέθηκε"
    print(f"  {status}  {name}: {p}")


In [ ]:

# ============================================================
# Προαιρετικός έλεγχος διαδρομών εισόδου
# ============================================================
if Path("/kaggle/input").exists():
    print("=== /kaggle/input/ πλήρης δομή ===")
    for root, dirs, files in os.walk("/kaggle/input"):
        level = root.replace("/kaggle/input", "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in sorted(files):
            size = os.path.getsize(os.path.join(root, f)) / (1024*1024)
            print(f"{indent}  - {f}  ({size:.1f} MB)")
else:
    print("Δεν υπάρχει /kaggle/input σε local εκτέλεση.")


In [ ]:

# ============================================================
# Κλειδί OpenAI API
# ============================================================

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
RUN_RAGAS = bool(OPENAI_API_KEY)

if not RUN_RAGAS:
    raise RuntimeError("OPENAI_API_KEY is required for canonical RAGAS evaluation.")

print("Το OPENAI_API_KEY εντοπίστηκε στο περιβάλλον εκτέλεσης.")


In [ ]:
# ============================================================
# Έξοδος QA CSVs
# ============================================================
qa_path_map = {
    "dense"           : DENSE_QA_PATH,
    "hybrid"          : HYBRID_QA_PATH,
    "hybrid_reranked" : RERANK_QA_PATH,
}

qa_dfs = {}
for run_name, path in qa_path_map.items():
    if path.exists():
        df = pd.read_csv(path)
        qa_dfs[run_name] = df
        print(f"OK: {run_name}: {df.shape}  |  columns: {df.columns.tolist()}")
    else:
        print(f"Σφάλμα: {run_name}: το αρχείο δεν βρέθηκε -> {path}")

if not qa_dfs:
    raise FileNotFoundError("Κανένα QA CSV δεν βρέθηκε. Έλεγξε τα paths παραπάνω.")


In [ ]:
# ============================================================
# Αυτόματη ανίχνευση στηλών ανά run
# (dense και hybrid CSVs έχουν διαφορετικά ονόματα στηλών)
# ============================================================
COL_MAPS = {}

for run_name, df in qa_dfs.items():
    cols = df.columns.tolist()

    # ερώτηση
    q_col = "question" if "question" in cols else None

    # παραγόμενη απάντηση
    ans_col = next((c for c in ["generated_answer", "answer"] if c in cols), None)

    # αναμενόμενη / gold απάντηση
    gt_col = next((c for c in ["expected_answer", "gold_answer", "answer_text"] if c in cols), None)

    # συμφραζόμενα
    ctx_col = next((c for c in ["context_text", "context", "contexts"] if c in cols), None)

    COL_MAPS[run_name] = {
        "question": q_col,
        "answer"  : ans_col,
        "gt"      : gt_col,
        "context" : ctx_col,
    }
    print(f"{run_name}: {COL_MAPS[run_name]}")

missing = [(r, k) for r, m in COL_MAPS.items() for k, v in m.items() if v is None]
if missing:
    print(f"\nΕλλιπείς στήλες: {missing}")
    print("Ελέγχεται το COL_MAPS πριν συνεχιστεί η αξιολόγηση.")
else:
    print("\nΌλες οι στήλες εντοπίστηκαν.")


In [ ]:

RAGAS_SETUP_ERROR = None

if RUN_RAGAS:
    try:
        from datasets import Dataset
        from ragas import evaluate
        from ragas.metrics import (
            context_precision,
            context_recall,
            faithfulness,
            answer_relevancy,
        )
        from ragas.llms import LangchainLLMWrapper
        from ragas.embeddings import LangchainEmbeddingsWrapper
        from langchain_openai import ChatOpenAI
        from langchain_huggingface import HuggingFaceEmbeddings

        ragas_llm = LangchainLLMWrapper(
            ChatOpenAI(model="gpt-4o-mini", temperature=0)
        )

        hf_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
        ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

        for metric in [context_precision, context_recall, faithfulness, answer_relevancy]:
            metric.llm = ragas_llm

        answer_relevancy.embeddings = ragas_embeddings

        METRICS = [context_precision, context_recall, faithfulness, answer_relevancy]
        print("Η ρύθμιση RAGAS ολοκληρώθηκε.")
    except Exception as exc:
        RAGAS_SETUP_ERROR = f"{type(exc).__name__}: {exc}"
        RUN_RAGAS = False
        METRICS = []
        print("Η ρύθμιση RAGAS απέτυχε. Θα γραφτούν placeholder metrics.")
        print(RAGAS_SETUP_ERROR)
else:
    Dataset = None
    METRICS = []


In [ ]:
if not RUN_RAGAS:
    raise RuntimeError(f"RAGAS setup failed: {RAGAS_SETUP_ERROR}")


In [ ]:

def build_ragas_dataset(run_name: str, qa_df: pd.DataFrame, sample_limit=None):
    df = qa_df.copy()
    if sample_limit is not None:
        df = df.head(sample_limit).copy()

    col = COL_MAPS[run_name]
    records = []
    for _, row in df.iterrows():
        ctx_raw = str(row.get(col["context"], ""))
        contexts = [c.strip() for c in ctx_raw.split("-" * 80) if c.strip()]
        if not contexts:
            contexts = [ctx_raw.strip()] if ctx_raw.strip() else ["no context"]

        records.append({
            "question"    : str(row.get(col["question"], "")),
            "answer"      : str(row.get(col["answer"], "")),
            "ground_truth": str(row.get(col["gt"], "")),
            "contexts"    : contexts,
        })

    if Dataset is None:
        return pd.DataFrame(records)
    return Dataset.from_pandas(pd.DataFrame(records))


In [ ]:

import nest_asyncio
import asyncio
import time

if RUN_RAGAS:
    from ragas import RunConfig

    nest_asyncio.apply()

    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

    run_config = RunConfig(
        timeout=180,
        max_retries=8,
        max_wait=60,
        max_workers=3
    )
else:
    run_config = None


def placeholder_ragas_scores(run_name: str, reason: str) -> dict:
    return {
        "run_name": run_name,
        "context_precision": 0.0,
        "context_recall": 0.0,
        "faithfulness": 0.0,
        "answer_relevancy": 0.0,
        "ragas_status": "skipped",
        "ragas_skip_reason": reason,
    }


ragas_rows = []
run_level_frames = []

for run_name in RUN_NAMES:
    if run_name not in qa_dfs:
        continue

    print(f"\nΑξιολόγηση run: {run_name}...")

    if not RUN_RAGAS:
        reason = RAGAS_SETUP_ERROR or "OPENAI_API_KEY not available"
        ragas_rows.append(placeholder_ragas_scores(run_name, reason))
        continue

    dataset = build_ragas_dataset(
        run_name,
        qa_dfs[run_name],
        sample_limit=RAGAS_SAMPLE_LIMIT
    )

    try:
        result = evaluate(
            dataset=dataset,
            metrics=METRICS,
            run_config=run_config,
            raise_exceptions=False
        )

        res_df = result.to_pandas()
        res_df.insert(0, "run_name", run_name)
        run_level_frames.append(res_df)

        numeric_scores = (
            res_df
            .select_dtypes(include=[np.number])
            .mean()
            .to_dict()
        )

        numeric_scores["run_name"] = run_name
        numeric_scores["ragas_status"] = "completed"
        numeric_scores["ragas_skip_reason"] = ""
        ragas_rows.append(numeric_scores)

        print(f"Ολοκληρώθηκε: {run_name}")
        display(pd.DataFrame([numeric_scores]))

        time.sleep(15)

    except Exception as e:
        reason = f"{type(e).__name__}: {e}"
        print(f"Σφάλμα στο {run_name}: {reason}")
        ragas_rows.append(placeholder_ragas_scores(run_name, reason))

ragas_results_df = pd.DataFrame(ragas_rows)

ragas_detailed_df = (
    pd.concat(run_level_frames, ignore_index=True)
    if run_level_frames
    else pd.DataFrame()
)

print("\nΜέσοι όροι ανά run:")
display(ragas_results_df)


In [ ]:
expected_runs = set(qa_dfs)
completed_runs = set(ragas_results_df.loc[ragas_results_df["ragas_status"] == "completed", "run_name"])
if completed_runs != expected_runs:
    missing_runs = sorted(expected_runs - completed_runs)
    raise RuntimeError(f"RAGAS evaluation incomplete for runs: {missing_runs}")


In [ ]:

# ============================================================
# Αποθήκευση
# ============================================================
if SAVE_OUTPUTS and len(ragas_results_df):
    ragas_results_df.to_csv(RAGAS_RESULTS_PATH, index=False, encoding="utf-8")
    print(f"Ολοκληρώθηκε: {RAGAS_RESULTS_PATH}")

if SAVE_OUTPUTS and len(ragas_detailed_df):
    ragas_detailed_df.to_csv(RAGAS_DETAILS_PATH, index=False, encoding="utf-8")
    print(f"Ολοκληρώθηκε: {RAGAS_DETAILS_PATH}")


In [ ]:

import shutil

if RAGAS_RESULTS_PATH.exists():
    archive_base = EVAL_DIR / "ragas_results"
    shutil.make_archive(str(archive_base), "zip", EVAL_DIR)
    print(f"Το αρχείο {archive_base}.zip δημιουργήθηκε επιτυχώς!")
else:
    print("Δεν δημιουργήθηκε ZIP γιατί δεν υπάρχει ragas_results.csv.")


## Τι σημαίνουν οι μετρικές

| Μετρική | Τι μετράει |
|---|---|
| **context_precision** | Πόσα από τα retrieved chunks είναι πραγματικά σχετικά με την ερώτηση |
| **context_recall** | Καλύπτει το retrieved context όλη την πληροφορία της gold απάντησης |
| **faithfulness** | Η παραγόμενη απάντηση βασίζεται αποκλειστικά στο context (0 = hallucination) |
| **answer_relevancy** | Πόσο σχετική είναι η απάντηση με την ερώτηση |
